In [ ]:
# foodrag bootstrap: run from repo root + load .env
from foodrag.notebook import setup
setup()

# 1. Setup

This notebook is based on the [Llama Index KG Tutorial](https://docs.llamaindex.ai/en/stable/examples/index_structs/knowlrelationship_graph/Neo4jKGIndexDemo/). More related topics:

- For the by llamaindex different supported graph stores see [here](https://docs.llamaindex.ai/en/stable/community/integrations/graph_stores/)
- [Neo4J GraphRAG](https://docs.llamaindex.ai/en/stable/examples/cookbooks/GraphRAG_v2/)
- [Neo4J LlamaIndex Integration](https://neo4j.com/labs/genai-ecosystem/llamaindex/)
- [Multimodal RAG with Neo4J](https://www.llamaindex.ai/blog/multimodal-rag-pipeline-with-llamaindex-and-neo4j-a2c542eb0206)

### Prerequisites:
Embeddings are already generated and stored in the `data/embeddings/` folder. If not, please run the `create_embeddings.ipynb` notebook.

### Steps:
1. Initial setup
2. Install & connect to Neo4J
3. Preprocess graph data
4. Store & load graph
5. Persist graph to Neo4J
6. Test connection to Neo4J


In [ ]:
from foodrag.hummus_transformation import transform_kg
from foodrag.utils import load_util
from foodrag.food_rag.utils.neo4J_helper import Neo4jManager

In [ ]:
# Output Paths
data_path = 'data'
embedding_path = data_path + '/embeddings/snowflake_l'

# Load key
api_key = load_util.load_file('config/api_key')

# 2. Install & connect to Neo4J
A) Windows Desktop:
1. Download Neo4J Desktop Client from [here](https://neo4j.com/download/)
2. Create new Project `Project`, init new the local DBMS `neo4j` with password `neo4jfoodrag` and start it.
3. Create the new databases `all`, `nousda`, `unlinkedhummus`, `onlyfoodon`, and `docs` (refer to transform_kg.py MODES and the db nam,e defined in init_graph_store_2.ipynb).
4. On the DBMS tab, go to plugins and install the `APOC` plugin.
5. Edit settings:
    - Set `dbms.security.allow_csv_import_from_file_urls=true`
    -  Comment out `server.directories.import=import`

B) Linux Remote:
 TODO

In [ ]:
# Define type of transformation
transform_mode = transform_kg.MODE_ALL

# Set up Neo4J params
username = "neo4j"
password = "neo4jfoodrag"
url = "bolt://localhost:7687"
database = transform_mode

# Connect to Neo4j
neo4j_kg = Neo4jManager(url, username, password, database)

# 3. Preprocess graph data

1. Load Data
2. Transform Data
3. Investigate

## 3.1 Load Data

In [ ]:
# Load the current recipes data
hummus_data = load_util.get_current_hummus(data_path)

## 3.2 Transform Data

In [ ]:
# Transform
kg_transformer = transform_kg.Neo4JTransformer(mode=transform_mode)
kg_transformer.transform(hummus_data, limit_size=1000)

## 3.3 Investigate

In [ ]:
# Show all node types
kg_transformer.get_all_node_types()

In [ ]:
# Show all relation types
kg_transformer.get_all_relation_types()

In [ ]:
# Show all vertices present in certain files
kg_transformer.get_all_node_file_dicts()

In [ ]:
# Show all relation present in certain files
kg_transformer.get_all_relation_file_dicts()

In [ ]:
# Show all node features
kg_transformer.get_all_node_features()

In [ ]:
# Show all node features as they appear in Neo4J
kg_transformer.get_all_cleaned_node_features()

In [ ]:
# Show all relation features
kg_transformer.get_all_relation_features()

In [ ]:
# Show all node and relation tables
all_objects = kg_transformer.get_all_objects()

for name, df in all_objects.items():
    print(name)
    display(df.head())
    print()

In [ ]:
# show all table colum names and types
all_objects = kg_transformer.get_all_objects()

for name, df in all_objects.items():
    print(name)
    print(df.dtypes)
    print()

# 4. Store and load the Graph

In [ ]:
# Store the graph
kg_transformer.save(data_path=data_path)

In [ ]:
# Load the graph
kg_transformer = transform_kg.Neo4JTransformer.load(data_path=data_path, mode=transform_mode)

# 5. Persist graph to Neo4J

In [ ]:
# Clear all nodes and relations
neo4j_kg.delete_all()

In [ ]:
# Insert into Neo4j (data needs to be stored first on path)
# Note: When only inserting test data relations might be missing as a matching source nodes might not be present and thus the corresponding relation will not be added. Especially affected: relations HAS_REVIEW and HAS_INGREDIENT.
neo4j_kg.insert_data(data_path=data_path, mode=transform_mode, batch_size=1000, verbose=True)

# 6. Test connection to Neo4J

In [ ]:
# Look at the first 5 recipes
neo4j_kg.query("MATCH (r:Recipe) RETURN r.title LIMIT 5")

In [ ]:
# Look at the first 5 ingredients
neo4j_kg.query("MATCH (i:Ingredient) RETURN i.name LIMIT 5")

In [ ]:
# Look at the first 5 relationship types
# Note: When only inserting test data relations might be missing as a matching source nodes might not be present and thus the corresponding relation will not be added. Especially affected: relations HAS_REVIEW and HAS_INGREDIENT.
neo4j_kg.query("MATCH (a)-[r]->(b) RETURN DISTINCT type(r) LIMIT 5")

In [ ]:
# Look at the first 5 object-subject pairs with relationship of type `SUB_CLASS_OF`
# Note: For the testing data when only a limited amount of nodes is inserted it might be thus the case that labels are missing.
neo4j_kg.query("MATCH (a)-[r:SUB_CLASS_OF]->(b) RETURN a.label, r, b.label LIMIT 5")